# Phenotype exploration -- data science for funsies

Reads directly from `MODELING_TABLE_DIR` (`residualize_phenotypes.ipynb`'s `prepare_modeling_tables()` output -- one TSV per phenotype, already pulled/filtered/covariate-joined) -- no BigQuery calls anywhere in this notebook, so it's cheap to rerun as often as you like. Four things:

1. Rank phenotypes by number of observations (post plausible-range + keep-list filter)
2. Raw vs. rank-inverse-normal-transformed density, per phenotype -- does the skew transform actually do anything?
3. `base` vs. each of the 3 richer covariate sets' residualized density (`base+PCs`, `base+PCs+zip3`, `base+PCs+zip3+SES`) + "sound statistics" (N retained, R², post-residual skewness) across the nested covariate-set staircase, reusing `residualize_lib.R`'s real `residualize_phenotype()` -- not a reimplementation
4. Pairwise correlations between phenotypes (on the rank-inverse-normal-transformed scale, so a skewed phenotype doesn't distort the correlation the way it would on the raw scale)

**Subsampling policy:** every skewness/R²/N-observation statistic below is computed on the *full* filtered table for that phenotype -- only plot rendering and the correlation join subsample to `MAX_PLOT_N`, since a density plot or scatter of 250k+ points buys nothing over 20k and costs a lot of render time. Same output-hygiene note as every other real-data notebook here: this pulls person-level rows into memory (never printed directly, only plotted/aggregated), so clear outputs / run `nbstripout` before committing, per root `README.md`.

**Plot sizing:** Jupyter's default R plot size is small for anything faceted across ~15-20 phenotypes. `options(repr.plot.width=, repr.plot.height=)` is set once to a bigger-than-default fallback in Setup, then overridden per plot (scaled to the actual number of phenotypes/facets loaded) right before each `ggplot()` call below -- adjust those numbers directly if a specific plot still looks cramped or too sparse.

## Setup

In [ ]:
required_pkgs <- c("dplyr", "readr", "e1071", "ggplot2")
missing_pkgs <- required_pkgs[!sapply(required_pkgs, requireNamespace, quietly = TRUE)]
if (length(missing_pkgs) > 0) install.packages(missing_pkgs)

library(dplyr)
library(readr)
library(ggplot2)
source("../../scripts/local/residualize_lib.R")   # skew_summary()/residualize_phenotype()/build_covariate_sets() -- shared with residualize_phenotypes.ipynb, not reimplemented here

theme_set(theme_minimal(base_size = 12))
options(repr.plot.width = 10, repr.plot.height = 7)   # bigger-than-default Jupyter plot size; individual
                                                       # cells below override this with a size that scales
                                                       # with the number of phenotypes/facets actually loaded

## Inputs

In [ ]:
WORKSPACE_BUCKET <- path.expand("~/workspace/Data from All of Us Controlled Tier /shared-env-pilot")
PHENOTYPE_BUCKET_DIR <- file.path(WORKSPACE_BUCKET, "data", "02_phenotype")
MODELING_TABLE_DIR <- file.path(PHENOTYPE_BUCKET_DIR, "modeling_tables")   # residualize_phenotypes.ipynb's prepare_modeling_tables() output

PHENO_LIST_PATH <- "../../docs/phenotype_list.tsv"
pc_cols <- paste0("PC", 1:5)   # top 5 only, same convention as residualize_phenotypes.ipynb

MAX_PLOT_N <- 20000   # subsample cap for plotting/joins only -- see subsampling policy above
RANDOM_SEED <- 0
set.seed(RANDOM_SEED)

subsample_df <- function(df, n = MAX_PLOT_N) {
  if (nrow(df) > n) df[sample(nrow(df), n), ] else df
}

## Load modeling tables

Tries every `phenotype_name` in `docs/phenotype_list.tsv`, not just the ones with a confirmed `concept_id` -- `hip_circumference`/`waist_hip_ratio` are still `UNCONFIRMED` as of writing, so their tables won't exist yet, but file-existence is a simpler and more robust "is this ready" check here than replicating `residualize_phenotypes.ipynb`'s `concept_id != "UNCONFIRMED"` filter logic (which doesn't even cleanly cover `waist_hip_ratio`'s `"UNCONFIRMED,UNCONFIRMED"` shape). Missing tables are skipped with a `message()`, same error-tolerant convention as `prepare_modeling_tables()`.

In [ ]:
pheno_list <- read_tsv(PHENO_LIST_PATH, col_types = cols(.default = "c"))

load_modeling_table <- function(name) {
  path <- file.path(MODELING_TABLE_DIR, paste0(name, ".tsv"))
  if (!file.exists(path)) {
    message(sprintf("Skipping '%s': no modeling table at %s", name, path))
    return(NULL)
  }
  read_tsv(path, show_col_types = FALSE)
}

pheno_tables <- setNames(lapply(pheno_list$phenotype_name, load_modeling_table), pheno_list$phenotype_name)
pheno_tables <- pheno_tables[!sapply(pheno_tables, is.null)]

sprintf("Loaded %d of %d phenotypes' modeling tables", length(pheno_tables), nrow(pheno_list))

## 1. Rank by number of observations

N after `filter_plausible_range()` + the keep-list restriction -- i.e. the same N `prepare_modeling_tables()` actually wrote to each phenotype's table, not a fresh count.

In [ ]:
n_obs_table <- tibble(
  phenotype_name = names(pheno_tables),
  n = sapply(pheno_tables, nrow)
) %>% arrange(desc(n))

n_obs_table

options(repr.plot.width = 10, repr.plot.height = max(6, 0.35 * nrow(n_obs_table) + 2))
ggplot(n_obs_table, aes(x = reorder(phenotype_name, n), y = n)) +
  geom_col(fill = "#3d5a80") +
  coord_flip() +
  labs(title = "Phenotypes ranked by number of observations", x = NULL,
       y = "N (post plausible-range filter + keep-list restriction)")

## 2. Skewness transform: raw vs. invnorm

`phenotype__invnorm` is already in each modeling table (written by `add_transformed_variant()` inside `prepare_modeling_tables()`) -- no retransforming here, just comparing what's already there. `skew_summary()` (from `residualize_lib.R`) on the full table; the density plot below subsamples per phenotype for rendering only.

In [ ]:
skew_summary_table <- bind_rows(lapply(names(pheno_tables), function(name) {
  df <- pheno_tables[[name]]
  bind_rows(
    skew_summary(df$phenotype, "raw"),
    skew_summary(df$phenotype__invnorm, "invnorm")
  ) %>% mutate(phenotype_name = name, .before = 1)
}))
skew_summary_table

In [ ]:
density_long <- bind_rows(lapply(names(pheno_tables), function(name) {
  df <- subsample_df(pheno_tables[[name]])
  bind_rows(
    tibble(phenotype_name = name, variant = "raw", value = df$phenotype),
    tibble(phenotype_name = name, variant = "invnorm", value = df$phenotype__invnorm)
  )
}))

n_facet_rows <- ceiling(length(pheno_tables) / 4)
options(repr.plot.width = 14, repr.plot.height = max(6, n_facet_rows * 2.5))
ggplot(density_long, aes(x = value, fill = variant)) +
  geom_density(alpha = 0.5, color = NA) +
  facet_wrap(~ phenotype_name, scales = "free", ncol = 4) +
  scale_fill_manual(values = c(raw = "#e07a5f", invnorm = "#3d5a80")) +
  labs(title = "Raw vs. rank-inverse-normal-transformed density, per phenotype", x = NULL, y = NULL) +
  theme(legend.position = "top", strip.text = element_text(size = 8))

## 3. Residualization: base vs. base+X, and R² across the covariate staircase

Fits all 4 of `build_covariate_sets()`'s nested covariate sets (`base` -> `base_pcs` -> `base_pcs_zip3` -> `base_pcs_zip3_ses`) once per phenotype, reusing `residualize_lib.R`'s real `residualize_phenotype()` -- same function `run_residualization_from_tables()` calls, not a reimplementation. All-NA covariate columns (zip3/SES not always populated for every phenotype's cohort) are dropped from that phenotype's model first, same guard `run_residualization_from_tables()` uses; a covariate-set left with nothing usable is skipped for that phenotype (`NULL`) rather than crashing `lm()` on an empty formula.

Compares **`base`'s residualized distribution against each of the 3 richer models**, not the raw phenotype -- raw vs. any residualized variant is an apples-to-oranges comparison (residualizing changes the scale/shape by construction), whereas `base` (age only) vs. `base+PCs`/`base+PCs+zip3`/`base+PCs+zip3+SES` isolates what each additional covariate block actually does on top of the minimal model. All 4 models get fit once, in the next cell, and reused for both the density comparison and the R² staircase plot below -- no redundant refitting.

In [ ]:
covariate_sets <- build_covariate_sets(pc_cols)

usable_covariates <- function(df, cols) {
  cols[sapply(cols, function(col) !all(is.na(df[[col]])))]
}

fit_all_models <- function(df) {
  setNames(lapply(names(covariate_sets), function(covset_name) {
    cols <- usable_covariates(df, covariate_sets[[covset_name]])
    if (length(cols) == 0) return(NULL)   # e.g. zip3/ses entirely NA for this phenotype's cohort
    residualize_phenotype(df, "phenotype", "sex_at_birth", cols)
  }), names(covariate_sets))
}

all_model_results <- lapply(pheno_tables, fit_all_models)

residual_stats_table <- bind_rows(lapply(names(all_model_results), function(name) {
  models <- all_model_results[[name]]
  bind_rows(lapply(names(models), function(covset_name) {
    r <- models[[covset_name]]
    if (is.null(r)) return(NULL)
    tibble(
      phenotype_name = name, covariate_set = covset_name,
      n_input = r$n_input, n_retained = r$n_retained,
      pct_retained = round(100 * r$n_retained / r$n_input, 1),
      r_squared = round(r$r_squared, 4),
      skewness_post = round(e1071::skewness(r$data$phenotype_norm, na.rm = TRUE), 3)
    )
  }))
}))
residual_stats_table$covariate_set <- factor(residual_stats_table$covariate_set, levels = names(covariate_sets))
residual_stats_table <- residual_stats_table %>% arrange(phenotype_name, covariate_set)

residual_stats_table

In [ ]:
covset_added_names <- setdiff(names(covariate_sets), "base")   # base_pcs, base_pcs_zip3, base_pcs_zip3_ses

residual_density_long <- bind_rows(lapply(names(all_model_results), function(name) {
  models <- all_model_results[[name]]
  if (is.null(models$base)) return(NULL)
  base_df <- subsample_df(models$base$data)
  bind_rows(lapply(covset_added_names, function(covset_name) {
    if (is.null(models[[covset_name]])) return(NULL)
    target_df <- subsample_df(models[[covset_name]]$data)
    bind_rows(
      tibble(phenotype_name = name, covariate_set = covset_name, role = "base", value = base_df$phenotype_norm),
      tibble(phenotype_name = name, covariate_set = covset_name, role = covset_name, value = target_df$phenotype_norm)
    )
  }))
}))
residual_density_long$covariate_set <- factor(residual_density_long$covariate_set, levels = covset_added_names)
residual_density_long$role <- factor(residual_density_long$role, levels = names(covariate_sets))

options(repr.plot.width = 12, repr.plot.height = max(8, 1.3 * length(pheno_tables)))
ggplot(residual_density_long, aes(x = value, fill = role)) +
  geom_density(alpha = 0.5, color = NA) +
  facet_grid(phenotype_name ~ covariate_set, scales = "free") +
  scale_fill_manual(values = c(base = "grey60", base_pcs = "#3d5a80",
                                base_pcs_zip3 = "#81b29a", base_pcs_zip3_ses = "#e07a5f")) +
  labs(title = "base vs. base+X residualized density, per phenotype", x = NULL, y = NULL) +
  theme(legend.position = "top", strip.text.y = element_text(angle = 0, size = 8), strip.text.x = element_text(size = 9))

### R² across the nested covariate-set staircase

Same `residual_stats_table` computed above -- no refitting -- how much of each successive block (PCs, zip3, SES) actually buys in variance explained, not just the full model's final R².

In [ ]:
n_facet_rows <- ceiling(length(pheno_tables) / 4)
options(repr.plot.width = 14, repr.plot.height = max(6, n_facet_rows * 2.5))
ggplot(residual_stats_table, aes(x = covariate_set, y = r_squared, group = 1)) +
  geom_line(color = "#3d5a80") +
  geom_point(color = "#3d5a80", size = 2) +
  facet_wrap(~ phenotype_name, ncol = 4) +
  labs(title = "R² as covariates are added (nested staircase, raw variant)", x = NULL, y = expression(R^2)) +
  theme(axis.text.x = element_text(angle = 30, hjust = 1), strip.text = element_text(size = 8))

### R² by phenotype, ordered by full-model R²

Same `residual_stats_table`, replotted with phenotypes on the x-axis instead of covariate-sets -- easier to compare phenotypes against each other than the faceted staircase above. Ordered by descending `base_pcs_zip3_ses` (full-model) R²; `covariate_set` is a genuine ordinal progression (`base` -> ... -> `base_pcs_zip3_ses`), so it's colored with a sequential (not categorical) scale.

In [ ]:
# order by descending full-model R^2 -- falls back to NA (sorted last) for any phenotype
# where base_pcs_zip3_ses was skipped (e.g. zip3/ses entirely NA for its cohort)
pheno_order_by_r2 <- residual_stats_table %>%
  filter(covariate_set == "base_pcs_zip3_ses") %>%
  arrange(desc(r_squared)) %>%
  pull(phenotype_name)
pheno_order_by_r2 <- union(pheno_order_by_r2, unique(residual_stats_table$phenotype_name))

residual_stats_table$phenotype_name <- factor(residual_stats_table$phenotype_name, levels = pheno_order_by_r2)

options(repr.plot.width = 14, repr.plot.height = 7)
ggplot(residual_stats_table, aes(x = phenotype_name, y = r_squared, color = covariate_set)) +
  geom_point(size = 3) +
  scale_color_viridis_d(option = "viridis") +   # sequential, since covariate_set is an ordered
                                                 # staircase (base -> ... -> base_pcs_zip3_ses),
                                                 # not an unordered category
  labs(title = "R² across covariate-set models, per phenotype (ordered by full-model R²)",
       x = NULL, y = expression(R^2), color = "Covariate set") +
  theme(axis.text.x = element_text(angle = 60, hjust = 1))

### R² by phenotype, ordered by base → full-model R² gain

Same plot shape as above, but ordered by how much R² each phenotype actually *gains* from `base` to `base_pcs_zip3_ses`, rather than the full model's absolute R² -- surfaces which phenotypes benefit most from adding PCs/zip3/SES on top of age alone, as distinct from which just happen to have the highest overall R² (a phenotype could have high absolute R² almost entirely from `base`, i.e. near-zero gain, and rank very differently here than in the previous plot).

In [ ]:
# delta = full-model R^2 minus base-model R^2, per phenotype -- a plain join (not pivot_wider,
# to avoid pulling in tidyr for one reshape) between the two covariate-set slices
base_r2 <- residual_stats_table %>% filter(covariate_set == "base") %>% select(phenotype_name, base = r_squared)
full_r2 <- residual_stats_table %>% filter(covariate_set == "base_pcs_zip3_ses") %>% select(phenotype_name, full = r_squared)
delta_r2_table <- base_r2 %>% inner_join(full_r2, by = "phenotype_name") %>% mutate(delta_r2 = full - base)

pheno_order_by_delta <- delta_r2_table %>%
  arrange(desc(delta_r2)) %>%
  pull(phenotype_name)
pheno_order_by_delta <- union(pheno_order_by_delta, unique(residual_stats_table$phenotype_name))

# reorders the same shared table's factor levels again -- fine since ggplot captures data
# at call time, but this cell's plot must run after this line, not before
residual_stats_table$phenotype_name <- factor(residual_stats_table$phenotype_name, levels = pheno_order_by_delta)

options(repr.plot.width = 14, repr.plot.height = 7)
ggplot(residual_stats_table, aes(x = phenotype_name, y = r_squared, color = covariate_set)) +
  geom_point(size = 3) +
  scale_color_viridis_d(option = "viridis") +   # sequential, since covariate_set is an ordered
                                                 # staircase (base -> ... -> base_pcs_zip3_ses),
                                                 # not an unordered category
  labs(title = "R² across covariate-set models, per phenotype (ordered by base -> full-model R² gain)",
       x = NULL, y = expression(R^2), color = "Covariate set") +
  theme(axis.text.x = element_text(angle = 60, hjust = 1))

## 4. Correlations between phenotypes

On the invnorm scale (already rank-based, so a skewed phenotype doesn't distort the correlation the way the raw scale would). Join cost is bounded regardless of cohort size: samples `MAX_PLOT_N` person_ids from the phenotype with the most observations, restricts every other phenotype's table to that same sample before joining -- `cor(..., use = "pairwise.complete.obs")` still only uses whichever pairs of phenotypes both have a non-NA value for a given person, so phenotypes with very different cohorts (e.g. `waist_circumference`) aren't penalized for not overlapping perfectly with the sampled ids.

In [ ]:
largest_pheno <- n_obs_table$phenotype_name[1]
sampled_ids <- pheno_tables[[largest_pheno]]$person_id
if (length(sampled_ids) > MAX_PLOT_N) sampled_ids <- sample(sampled_ids, MAX_PLOT_N)

wide_invnorm <- Reduce(function(acc, name) {
  col <- pheno_tables[[name]] %>%
    filter(person_id %in% sampled_ids) %>%
    transmute(person_id, !!name := phenotype__invnorm)
  full_join(acc, col, by = "person_id")
}, names(pheno_tables), tibble(person_id = sampled_ids))

cor_mat <- cor(wide_invnorm %>% select(-person_id), use = "pairwise.complete.obs")

cor_long <- as.data.frame(as.table(cor_mat)) %>%
  rename(phenotype_x = Var1, phenotype_y = Var2, correlation = Freq)

plot_size <- max(8, 0.45 * length(pheno_tables) + 3)
options(repr.plot.width = plot_size, repr.plot.height = plot_size)
ggplot(cor_long, aes(x = phenotype_x, y = phenotype_y, fill = correlation)) +
  geom_tile() +
  scale_fill_gradient2(low = "#3d5a80", mid = "white", high = "#e07a5f", midpoint = 0, limits = c(-1, 1)) +
  labs(title = "Pairwise phenotype correlations (invnorm scale)", x = NULL, y = NULL) +
  theme(axis.text.x = element_text(angle = 60, hjust = 1))

## Summary

`n_obs_table` -- which phenotypes are actually high-completeness vs. barely-usable (compare against `waist_circumference`'s known N=22 problem). `skew_summary_table` -- which phenotypes needed the invnorm transform at all (near-0 raw skewness means it didn't do much) vs. which it clearly fixed. `residual_stats_table`/the base-vs-base+X density grid/the staircase plot -- which covariates are actually buying variance explained per phenotype (a flat staircase past `base`, or a base+X panel that looks nearly identical to `base`, means zip3/SES aren't adding much for that trait) and whether `pct_retained` looks reasonable (a low value means the 5-SD post-residual trim is cutting hard, worth a second look). `cor_long`/the heatmap -- expected clusters (e.g. HDL/LDL/total cholesterol/triglycerides should correlate; height and hemoglobin shouldn't) are a sanity check on the pipeline more than a discovery in themselves at this stage.